# Imports

In [1]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate openai httpx==0.27.2

In [2]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
#from google.colab import drive
from huggingface_hub import login
#from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc
import gradio as gr

# Device

In [3]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")

if device == "cuda":
    !nvidia-smi

Device: cuda


# Constants

In [4]:
AUDIO_MODEL = "whisper-1"
LLAMA = "meta-llama/Meta-Llama-3-8B-Instruct"

# Audio File

In [5]:
audio_filename = "denver_extract.mp3"

# Initialization

In [6]:
from dotenv import load_dotenv
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
hf_api_key = os.getenv('HF_TOKEN')
openai = OpenAI()

# Whisper Agent

In [7]:
def transcribe(audio_file_path):
    try:
        # Debugging: Check if the file path is None or empty
        if audio_file_path is None or audio_file_path == "":
            return ""
        
        if not os.path.exists(audio_file_path):
            return f"Error: Audio file does not exist at path {audio_file_path}"

        # Check if file is empty
        if os.path.getsize(audio_file_path) == 0:
            return "Error: Audio file is empty"

        with open(audio_file_path, "rb") as audio_file:
            response = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")        
            # Debugging: Check the response object structure
            #print(f"OpenAI API response: {response.text}")
            # Extract the transcription text from the response
            return response
    except Exception as e:
        return f"An error occurred: {e}"

# Meeting-Minutes Agent

In [ ]:
# Top-level (global)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config
)
model.eval()

# Inside your function
def meeting_minutes(audio_file, device=device):

    transcription = transcribe(audio_file)

    system_message = "You are an assistant that produces minutes of meetings from transcripts, with summary, key discussion points, takeaways and action items with owners, in markdown."
    user_prompt = f"Below is an extract transcript of a Denver council meeting. Please write minutes in markdown, including a summary with attendees, location and date; discussion points; takeaways; and action items with owners.\n{transcription}"
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    input = tokenizer.apply_chat_template(messages, return_tensors="pt").to(device)
    streamer = TextStreamer(tokenizer)

    output = model.generate(input, max_new_tokens=2000, streamer=streamer)

    del inputs, streamer
    gc.collect()
    torch.cuda.empty_cache()

    return output

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 21.0M/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   3%|2         | 31.5M/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 21.0M/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   1%|          | 31.5M/5.00G [00:00<?, ?B/s]

In [13]:
# Inside your function
def meeting_minutes_api(audio_file):

    transcription = transcribe(audio_file)

    system_message = "You are an assistant that produces minutes of meetings from transcripts, with summary, key discussion points, takeaways and action items with owners, in markdown."
    user_prompt = f"Below is an extract transcript of a Denver council meeting. Please write minutes in markdown, including a summary with attendees, location and date; discussion points; takeaways; and action items with owners.\n{transcription}"

    full_prompt = f"<s>[INST] <<SYS>>{system_message}<</SYS>>\n{user_prompt} [/INST]"

    headers = {
        "Authorization": f"Bearer {hf_api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "inputs": full_prompt,
        "parameters": {
            "max_new_tokens": 2048,
            "temperature": 0.7,
            "return_full_text": False
        }
    }

    response = requests.post(
        "https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.1", #"https://api-inference.huggingface.co/models/gpt2", #"https://api-inference.huggingface.co/models/meta-llama/Meta-Llama-3-8B",
        headers=headers,
        json=payload
    )

    if response.status_code == 200:
        return response.json()[0]["generated_text"]
    else:
        return f"Error: {response.status_code} - {response.text}"

In [14]:
# Interface
with gr.Blocks(css="""
    .translation-box {
        height: 500px;
        overflow-y: auto;
        border: 1px solid var(--block-border-color);
        border-radius: var(--block-radius);
        padding: 10px;
        background-color: var(--block-background-fill);
    }
    """) as ui:
    gr.Markdown("## 🛫 MinutesAI Assistant\n")

    with gr.Row():
        mp3_input = gr.Audio(type="filepath", label="Upload MP3")
        text_output = gr.Textbox(label="Generated Meeting Minutes", lines=30)

        mp3_input.change(fn=meeting_minutes_api, inputs=mp3_input, outputs=text_output)
    
ui.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Missing file: C:\Users\ssre_\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\ssre_\.cache\huggingface\gradio\frpc
